In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 49.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [bitsandbytes]

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Torch: 2.10.0+cu128
Transformers: 5.0.0
PEFT: 0.18.1
TRL: 0.29.0
BitsAndBytes: 0.49.2


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

dataset = load_dataset(
    "json",
    data_files="/content/sql_d.json"
)



training_args = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    fp16=False,
    bf16=False
)




trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained("sql_model")
tokenizer.save_pretrained("sql_model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,3.125532
20,2.018083
30,1.392311
40,0.911191
50,0.796160
60,0.727746


('sql_model/tokenizer_config.json',
 'sql_model/chat_template.jinja',
 'sql_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/content/sql_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

prompt = "List all employees"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

List all employees' names and their job titles.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# base model
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# trained adapter
adapter_path = "/content/sql_model"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# load base model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    dtype=torch.float16
)

model = PeftModel.from_pretrained(model, adapter_path)

question = "list all complaints"

prompt = f"Question: {question} SQL:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0
)

result = tokenizer.decode(output[0], skip_special_tokens=True)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Question: list all complaints SQL: SELECT * FROM complaints_fluentgrid;


In [ ]:
!zip -r sql_model.zip /content/sql_model

In [ ]:
from google.colab import files
files.download("sql_model.zip")

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
!git clone https://github.com/ggerganov/llama.cpp
# %cd llama.cpp
!pip install -r requirements.txt

In [ ]:
import torch, torchvision

print(torch.__version__)
print(torchvision.__version__)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/sql_model"   # your trained LoRA path

# Load base model
model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16)

# Load LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

# 🔥 Merge LoRA
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("/content/merged_sql_model")

# Save tokenizer (VERY IMPORTANT)
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.save_pretrained("/content/merged_sql_model")

print("✅ Merge completed")


In [ ]:
!ls /content/merged_sql_model

In [ ]:
from transformers import AutoTokenizer

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=False)

tokenizer.save_pretrained("/content/merged_sql_model")

print("✅ Tokenizer fixed")


In [ ]:
!ls /content/merged_sql_model

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_sql_model --outfile /content/sql_model.gguf

In [ ]:
from google.colab import files
files.download("/content/sql_model.gguf")